# Aula 11 — DataFrames e leitura de arquivos CSV

**Módulo 4 — Pandas e Análise de Dados**

## Objetivos da aula

- Conhecer o Pandas e as estruturas `Series` e `DataFrame`.
- Criar e salvar um arquivo `.csv`, e depois lê-lo com o Pandas.
- Visualizar e inspecionar as primeiras informações de uma base de dados.

---

## 1. O que é o Pandas

O **Pandas** é a principal biblioteca Python para análise de dados tabulares — dados organizados em linhas e colunas, como uma planilha. Ele é construído sobre o NumPy (Módulo 3) e adiciona rótulos, nomes de colunas e várias ferramentas de leitura, filtragem e resumo de dados.

A partir desta aula, vamos trabalhar com uma base de dados que representa o **monitoramento de sensores de equipamentos industriais** — o cenário que acompanha o curso até a última aula, quando construiremos um modelo de Machine Learning sobre esses mesmos dados.

In [1]:
import numpy as np
import pandas as pd

print("Pandas importado com sucesso, versão:", pd.__version__)


Pandas importado com sucesso, versão: 2.2.1


## 2. `Series`: uma coluna com rótulos

Uma `Series` é um array unidimensional (como os do NumPy), mas com um **índice** rotulado para cada posição.

In [2]:
temperaturas = pd.Series([72.0, 76.5, 91.0, 68.3], name="temperatura")
print(temperaturas)
print("\nTipo:", type(temperaturas))


0    72.0
1    76.5
2    91.0
3    68.3
Name: temperatura, dtype: float64

Tipo: <class 'pandas.core.series.Series'>


## 3. `DataFrame`: uma tabela completa

Um `DataFrame` é a estrutura central do Pandas: uma **tabela** onde cada coluna é uma `Series`, todas compartilhando o mesmo índice de linhas. Podemos pensar nele como uma planilha do Excel manipulável por código.

Vamos criar nossa base de dados simulada de monitoramento industrial, com leituras de vários equipamentos.

In [3]:
np.random.seed(42)   # fixa a semente aleatória: os mesmos números "aleatórios" toda vez que rodarmos

n = 500   # quantidade de leituras simuladas
tipos_equipamento = ["Motor", "Bomba", "Compressor", "Ventilador"]

dados = pd.DataFrame({
    "equipamento_id": [f"EQ-{i:04d}" for i in range(1, n + 1)],
    "tipo_equipamento": np.random.choice(tipos_equipamento, size=n),
    "temperatura": np.round(np.random.normal(70, 12, size=n), 1),      # °C
    "pressao": np.round(np.random.normal(5.5, 1.3, size=n), 2),        # bar
    "vibracao": np.round(np.random.normal(2.4, 1.1, size=n), 2),       # mm/s
    "horas_operacao": np.random.randint(0, 10000, size=n),
})


def definir_status(linha):
    critico = (linha["temperatura"] >= 90) or (linha["vibracao"] >= 4.5) or (linha["pressao"] >= 8) or (linha["pressao"] <= 2)
    alerta = (linha["temperatura"] >= 80) or (linha["vibracao"] >= 3.5) or (linha["pressao"] >= 7) or (linha["pressao"] <= 3)
    if critico:
        return "critico"
    elif alerta:
        return "alerta"
    else:
        return "normal"


dados["status"] = dados.apply(definir_status, axis=1)

print("Base de dados criada com", len(dados), "linhas.")
dados.head()


Base de dados criada com 500 linhas.


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
0,EQ-0001,Compressor,59.8,6.42,3.01,1958,normal
1,EQ-0002,Ventilador,51.8,6.08,1.33,6344,normal
2,EQ-0003,Motor,64.6,5.03,2.52,5779,normal
3,EQ-0004,Compressor,80.3,7.01,0.93,6144,alerta
4,EQ-0005,Compressor,72.6,4.09,1.74,5063,normal


**Adicional para fixar a ideia dos eixos**
Em um DataFrame temos dois eixos:
```
                 axis=1 →
             colunas
          ┌─────┬─────┬─────┐
axis=0 ↓  │     │     │     │
          ├─────┼─────┼─────┤
 linhas   │     │     │     │
          ├─────┼─────┼─────┤
          │     │     │     │
          └─────┴─────┴─────┘
```

## 4. Salvando e lendo um arquivo CSV

**CSV** (*Comma-Separated Values*) é o formato mais comum para troca de dados tabulares: um arquivo de texto simples em que cada linha é um registro e as colunas são separadas por vírgula.

Vamos salvar nosso `DataFrame` em um arquivo CSV com `to_csv()`, e depois lê-lo de volta com `read_csv()` — simulando o fluxo real de trabalho: alguém (ou algum sistema) gera um arquivo de dados, e nós o carregamos para análise.

In [4]:
dados.to_csv("sensores_industriais.csv", index=False)   # index=False evita salvar o índice como coluna extra

print("Arquivo salvo!")


Arquivo salvo!


In [5]:
df = pd.read_csv("sensores_industriais.csv")

print(f"Base carregada: {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head()   # mostra as 5 primeiras linhas por padrão


Base carregada: 500 linhas e 7 colunas.


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
0,EQ-0001,Compressor,59.8,6.42,3.01,1958,normal
1,EQ-0002,Ventilador,51.8,6.08,1.33,6344,normal
2,EQ-0003,Motor,64.6,5.03,2.52,5779,normal
3,EQ-0004,Compressor,80.3,7.01,0.93,6144,alerta
4,EQ-0005,Compressor,72.6,4.09,1.74,5063,normal


> **No Google Colab:** se você quiser ler um CSV enviado por você (em vez de gerado no próprio notebook), pode fazer upload pelo painel de arquivos à esquerda, ou montar seu Google Drive com `from google.colab import drive; drive.mount('/content/drive')`.

## 5. Primeiras inspeções de um DataFrame

Antes de qualquer análise, é essencial "conhecer" os dados: quantas linhas e colunas existem, quais os tipos de cada coluna, e como são os primeiros e últimos registros.

In [6]:
df.head(3)     # 3 primeiras linhas


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
0,EQ-0001,Compressor,59.8,6.42,3.01,1958,normal
1,EQ-0002,Ventilador,51.8,6.08,1.33,6344,normal
2,EQ-0003,Motor,64.6,5.03,2.52,5779,normal


In [7]:
df.tail(3)     # 3 últimas linhas


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
497,EQ-0498,Ventilador,78.6,6.72,3.71,2780,alerta
498,EQ-0499,Motor,67.1,4.22,1.89,7398,normal
499,EQ-0500,Ventilador,65.5,5.21,2.62,1588,normal


In [8]:
print("Formato (linhas, colunas):", df.shape)
print("\nNomes das colunas:", list(df.columns))


Formato (linhas, colunas): (500, 7)

Nomes das colunas: ['equipamento_id', 'tipo_equipamento', 'temperatura', 'pressao', 'vibracao', 'horas_operacao', 'status']


In [9]:
df.info()      # tipo de cada coluna, quantidade de valores não nulos e uso de memória


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   equipamento_id    500 non-null    object 
 1   tipo_equipamento  500 non-null    object 
 2   temperatura       500 non-null    float64
 3   pressao           500 non-null    float64
 4   vibracao          500 non-null    float64
 5   horas_operacao    500 non-null    int64  
 6   status            500 non-null    object 
dtypes: float64(3), int64(1), object(3)
memory usage: 27.5+ KB


In [10]:
df.dtypes      # apenas os tipos de dado de cada coluna


equipamento_id       object
tipo_equipamento     object
temperatura         float64
pressao             float64
vibracao            float64
horas_operacao        int64
status               object
dtype: object

## 6. Resumo da aula

- O Pandas organiza dados tabulares em `DataFrame` (tabela) e `Series` (coluna), ambos construídos sobre o NumPy.
- `to_csv()` salva um `DataFrame` em arquivo; `read_csv()` faz o caminho inverso.
- `head()`, `tail()`, `shape`, `columns`, `info()` e `dtypes` são os primeiros comandos para "conhecer" uma base de dados nova.
- A partir desta aula, usaremos a base `sensores_industriais.csv` (recriada em cada notebook) em todo o restante do curso.

### Exercício sugerido

Depois de carregar `df`, use `df.sample(5)` (em vez de `head`) para visualizar 5 linhas aleatórias da base, e `df["tipo_equipamento"].unique()` para listar os tipos de equipamento presentes.
